# Objective 1c: layoff discussion on r/jobs vs layoffs.fyi

Checks whether layoff talk on r/jobs moves with reported layoffs, the external check in the proposal. Neither K-means nor LDA found a separate layoff topic, so the r/jobs side is measured with keywords ("laid off" and "layoff(s)", rules in `functions/layoff_mentions.py`):
- **Full archive**: every r/jobs comment in the Arctic Shift archive that matches, collected by `scripts/collect_layoff_mentions.py`, per 1,000 comments in the subreddit that month.
- **Part 2 sample**: the same rule on the 1,000-comment monthly sample, to check that the sample agrees with the full archive.
- **LDA T1** (K = 10, job-search difficulty, hiring process and AI): the topic that layoff comments lean towards most, as a reference curve.

The external side is layoffs.fyi (Kaggle mirror `swaptr/layoffs-2022`): reported layoff events and employees laid off per month. Only the 9 complete months, January–September 2026, are compared, so correlations are descriptive.

To reproduce the quarterly comparisons and original-unit figures from existing data, start at section 7 in a fresh kernel.

## 1. Setup

In [ ]:
# Add the repo root to Python's search path
import sys
sys.path.append('..')

from functions.layoff_mentions import layoff_terms

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

MENTIONS_DIR = Path('../dataset/reddit/layoff_mentions')
LAYOFFS_FILE = Path('../dataset/layoffs/layoffs.csv')
TOPICS_DIR = Path('../dataset/monthly_Topics')
FIG_DIR = MENTIONS_DIR / 'figures'
TERMS = ['laid_off', 'layoff']
COMPLETE = [f'2026-{m:02d}' for m in range(1, 10)]

## 2. Layoff mentions in the full r/jobs archive
For each month: the archive's total r/jobs comments, the comments the archive search returned for each term, and those that pass the exact rule. A comment that uses both terms counts once. Precision (exact / returned) shows how loose the archive search is: "laid off" also returns "laid back" and "laid out". October is cut off on 28 October, so it has no rate.

In [ ]:
run = json.loads((MENTIONS_DIR / 'run.json').read_text(encoding='utf-8'))
counts = pd.read_csv(MENTIONS_DIR / 'monthly_counts.csv').set_index('month')
counts['per_1000'] = counts['any_term_exact'] / counts['archive_comments'] * 1000
counts.loc[counts['partial_month'], 'per_1000'] = np.nan
for term in TERMS:
    counts[f'{term}_precision'] = counts[f'{term}_exact'] / counts[f'{term}_hits']
print('Collection finished', run['finished_utc'], '| archive queries:', run['queries'])
counts[['archive_comments', 'laid_off_exact', 'layoff_exact', 'any_term_exact', 'per_1000',
        'laid_off_precision', 'layoff_precision']].round(3)

## 3. Does Part 2's sample agree with the full archive?
The same rule on the 1,000 sampled comments per month, with a 95% Wilson interval. If the archive rate falls inside most intervals, the sample is a fair picture of the subreddit for this kind of content. The sample excludes deleted, bot and very short comments; the archive totals do not, which pulls the archive rate slightly down.

In [ ]:
corpus = pd.read_csv(TOPICS_DIR / 'comments_clean.csv', dtype={'comment_id': str}, keep_default_na=False)
corpus['mentions'] = corpus['text'].map(lambda text: bool(layoff_terms(text, TERMS)))
sample = corpus.groupby('month')['mentions'].agg(['sum', 'size'])


def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


low, high = wilson(sample['sum'], sample['size'])
check = pd.DataFrame({
    'sample_mentions': sample['sum'],
    'sample_per_1000': sample['sum'] / sample['size'] * 1000,
    'sample_low': low * 1000, 'sample_high': high * 1000,
    'archive_per_1000': counts['per_1000'],
})
check['inside_interval'] = check['archive_per_1000'].between(check['sample_low'], check['sample_high'])
print(f"Archive rate inside the sample interval in {check.loc[COMPLETE, 'inside_interval'].sum()} of {len(COMPLETE)} complete months")
check.round(1)

## 4. layoffs.fyi
Reported layoff events, distinct companies and employees laid off per month, overall and for the United States, where most r/jobs users work. Many events have no headcount, and a few very large cuts dominate some months (largest events below), so the number of events is the steadier measure.

In [ ]:
layoffs = pd.read_csv(LAYOFFS_FILE)
layoffs['date'] = pd.to_datetime(layoffs['date'], format='%m/%d/%Y')
layoffs = layoffs[layoffs['date'] >= '2026-01-01'].assign(month=lambda d: d['date'].dt.strftime('%Y-%m'))
us = layoffs['country'] == 'United States'
by_month = layoffs.groupby('month')
fyi = pd.DataFrame({
    'events': by_month.size(),
    'companies': by_month['company'].nunique(),
    'laid_off': by_month['total_laid_off'].sum(),
    'missing_headcount': by_month['total_laid_off'].apply(lambda s: s.isna().mean()),
    'us_events': layoffs[us].groupby('month').size(),
    'us_laid_off': layoffs[us].groupby('month')['total_laid_off'].sum(),
})
print('Latest event in the file:', layoffs['date'].max().date())
display(fyi.round(2))
layoffs.nlargest(5, 'total_laid_off')[['company', 'date', 'total_laid_off', 'country', 'industry']]

## 5. Comparison
Spearman correlations over the 9 complete months, for the same month and with r/jobs one month later than layoffs.fyi (8 pairs). With so few points, |ρ| has to reach about 0.7 before it is distinguishable from chance, and 18 correlations are computed, so one or two with p < 0.05 are expected by chance alone. Read the table as a description of how the curves move, not as evidence of a relationship. Raw mention counts also follow the overall fall in r/jobs activity, so compare the rate per 1,000 comments rather than counts. In the figure, every curve is standardised (z-score over January–September) so they share one axis.

In [ ]:
lda_shares = pd.read_csv(TOPICS_DIR / 'lda' / 'lda_monthly_shares.csv')
lda_t1 = lda_shares[lda_shares['topic'] == 1].set_index('month')['seed_42'] * 100

series = pd.DataFrame({
    'r/jobs mentions per 1,000 comments': counts['per_1000'],
    'r/jobs mentions': counts['any_term_exact'],
    'LDA T1 share (%)': lda_t1,
    'layoffs.fyi events': fyi['events'],
    'layoffs.fyi employees laid off': fyi['laid_off'],
    'layoffs.fyi US events': fyi['us_events'],
}).loc[COMPLETE]
display(series.round(2))

reddit = ['r/jobs mentions per 1,000 comments', 'r/jobs mentions', 'LDA T1 share (%)']
external = ['layoffs.fyi events', 'layoffs.fyi employees laid off', 'layoffs.fyi US events']
rows = []
for r in reddit:
    for e in external:
        same = spearmanr(series[r], series[e])
        lagged = spearmanr(series[r].iloc[1:].to_numpy(), series[e].iloc[:-1].to_numpy())
        rows.append({'r/jobs': r, 'layoffs.fyi': e,
                     'rho_same_month': same.statistic, 'p_same_month': same.pvalue,
                     'rho_rjobs_1_month_later': lagged.statistic, 'p_1_month_later': lagged.pvalue})
correlations = pd.DataFrame(rows)
correlations.round(3)

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
z = (series - series.mean()) / series.std()
x = range(len(COMPLETE))
ticks = [m[5:] for m in COMPLETE]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for column, style in [('r/jobs mentions per 1,000 comments', '-o'), ('LDA T1 share (%)', ':s'),
                      ('layoffs.fyi events', '--^'), ('layoffs.fyi employees laid off', '-.d')]:
    axes[0].plot(x, z[column], style, label=column)
axes[0].axhline(0, color='grey', linewidth=0.5)
axes[0].set_xticks(list(x), ticks)
axes[0].set(xlabel='2026 month', ylabel='z-score (Jan–Sep)', title='Layoff talk on r/jobs vs reported layoffs')
axes[0].legend(fontsize=8)

# both axes start at 0 so the fall in layoff mentions is not exaggerated
volume = counts.loc[COMPLETE, 'archive_comments'] / 1000
mentions = counts.loc[COMPLETE, 'any_term_exact']
axes[1].bar(x, volume, color='#c9d6e8', label='all r/jobs comments (thousands)')
axes[1].set(xlabel='2026 month', ylabel='All comments (thousands)', ylim=(0, volume.max() * 1.35),
            title='r/jobs volume and layoff mentions (full archive)')
axes[1].set_xticks(list(x), ticks)
twin = axes[1].twinx()
twin.plot(x, mentions, '-o', color='#d4553b', label='comments mentioning layoffs')
twin.set(ylabel='Comments mentioning layoffs', ylim=(0, mentions.max() * 1.35))
bars, bar_labels = axes[1].get_legend_handles_labels()
line, line_labels = twin.get_legend_handles_labels()
axes[1].legend(bars + line, bar_labels + line_labels, loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'layoffs_comparison.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(x, check.loc[COMPLETE, 'sample_per_1000'],
            yerr=[check.loc[COMPLETE, 'sample_per_1000'] - check.loc[COMPLETE, 'sample_low'],
                  check.loc[COMPLETE, 'sample_high'] - check.loc[COMPLETE, 'sample_per_1000']],
            fmt='o', capsize=4, label='Part 2 sample (95% interval)')
ax.plot(x, check.loc[COMPLETE, 'archive_per_1000'], '-s', label='full archive')
ax.set_xticks(list(x), ticks)
ax.set(xlabel='2026 month', ylabel='Mentions per 1,000 comments', title='Sample vs full archive')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'sample_vs_archive.png', dpi=150)
plt.show()

## 6. Export
Writes the monthly table, the sample check and the correlations to `dataset/reddit/layoff_mentions/` (ignored by git), next to the collected counts.

In [ ]:
series.join(fyi[['companies', 'missing_headcount', 'us_laid_off']]).to_csv(MENTIONS_DIR / 'comparison_monthly.csv')
check.to_csv(MENTIONS_DIR / 'sample_vs_archive.csv')
correlations.to_csv(MENTIONS_DIR / 'comparison_correlations.csv', index=False)
print('Saved to', MENTIONS_DIR)

## 7. Quarterly comparison and external context

This section combines three existing outputs: the full-archive keyword counts,
the layoffs.fyi event records and the monthly LDA topic weights. The purpose is
to compare changes in explicit layoff discussion with the external records and
with the broader topic composition identified by LDA.

Use the nine complete months from January to September. Q1 is January–March and
Q3 is July–September; partial October is excluded. Quarterly rates and topic
weights are calculated as the mean of the three monthly values, giving every
month equal weight.

The keyword rate and the LDA weights measure different content and use different
data scopes. The keyword rate counts explicit layoff terms in the full archive,
whereas T9 includes a broader set of economic and unemployment content in the
LDA sample. Their comparison is descriptive and should not be treated as an
independent validation test.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, StrMethodFormatter
from scipy.stats import spearmanr
from IPython.display import display

comparison_root = Path.cwd()
if comparison_root.name == 'notebook':
    comparison_root = comparison_root.parent
comparison_mentions_dir = comparison_root / 'dataset/reddit/layoff_mentions'
comparison_topics_dir = comparison_root / 'dataset/monthly_Topics'
comparison_output_dir = comparison_mentions_dir / 'analysis'
comparison_fig_dir = comparison_mentions_dir / 'figures'
comparison_output_dir.mkdir(parents=True, exist_ok=True)
comparison_fig_dir.mkdir(parents=True, exist_ok=True)
complete_months = [f'2026-{month:02d}' for month in range(1, 10)]
q1_months, q3_months = complete_months[:3], complete_months[6:9]

archive_monthly = pd.read_csv(comparison_mentions_dir / 'monthly_counts.csv').set_index('month')
archive_complete = archive_monthly.loc[complete_months].copy()
assert archive_complete['partial_month'].astype(str).str.lower().eq('false').all()
assert (archive_complete['archive_comments'] > 0).all()
assert archive_complete['any_term_exact'].between(0, archive_complete['archive_comments']).all()
archive_complete['mentions_per_1000'] = (
    archive_complete['any_term_exact'] / archive_complete['archive_comments'] * 1000
)

external_records = pd.read_csv(comparison_root / 'dataset/layoffs/layoffs.csv')
external_records['date'] = pd.to_datetime(external_records['date'], format='%m/%d/%Y')
external_records['total_laid_off'] = pd.to_numeric(external_records['total_laid_off'], errors='raise')
assert external_records['total_laid_off'].dropna().ge(0).all()
external_records['month'] = external_records['date'].dt.strftime('%Y-%m')
external_complete = external_records.loc[external_records['month'].isin(complete_months)]
external_grouped = external_complete.groupby('month')['total_laid_off']
external_monthly = pd.DataFrame({
    'events': external_grouped.size(),
    'known_events': external_grouped.count(),
    'disclosed_headcount': external_grouped.sum(min_count=1),
}).reindex(complete_months)
assert external_monthly.notna().all().all(), 'External coverage is incomplete.'
external_monthly['missing_events'] = external_monthly['events'] - external_monthly['known_events']
external_monthly['missing_headcount_pct'] = external_monthly['missing_events'] / external_monthly['events'] * 100
external_monthly['mean_disclosed_size'] = external_monthly['disclosed_headcount'] / external_monthly['known_events']

lda_context_run = json.loads((comparison_topics_dir / 'lda/lda_run.json').read_text(encoding='utf-8'))
assert hashlib.sha256((comparison_topics_dir / 'comments_clean.csv').read_bytes()).hexdigest() == lda_context_run['input_sha256']
assert lda_context_run['k'] == 10 and lda_context_run['seed'] == 42
lda_context = pd.read_csv(comparison_topics_dir / 'lda/lda_monthly_shares.csv')
context_weights = lda_context.pivot(index='month', columns='topic', values='seed_42').loc[complete_months]
assert context_weights.notna().all().all()
comparison_monthly = archive_complete[['archive_comments', 'any_term_exact', 'mentions_per_1000']].join(external_monthly)
comparison_monthly['t2_weight_pct'] = context_weights[2] * 100
comparison_monthly['t9_weight_pct'] = context_weights[9] * 100
display(comparison_monthly.round(3))
archive_total = int(archive_complete['archive_comments'].sum())
sample_total = len(pd.read_csv(comparison_topics_dir / 'lda/lda_comment_topics.csv', usecols=['comment_id']))
print(f'Archive comments, January–September: {archive_total:,}')
print(f'LDA sample, January–October: {sample_total:,}; modelled: {lda_context_run["modelled_comments"]:,}')
comparison_monthly.to_csv(comparison_output_dir / 'monthly_layoff_context.csv')

### 7.1 Measuring change from Q1 to Q3

Apply the same quarterly averaging rule to the archive mention rate, recorded
event count, disclosed headcount and selected LDA topic weights. For every
measure, calculate relative change as
`100 × (Q3 monthly mean / Q1 monthly mean − 1)`.

The calculation uses full-precision monthly values and rounds only the displayed
table and chart labels. This keeps the percentage changes consistent across
measures with different units.

In [ ]:
quarter_metrics = ['mentions_per_1000', 'disclosed_headcount', 'events', 't2_weight_pct', 't9_weight_pct']
quarter_summary = pd.DataFrame({
    'q1_monthly_mean': comparison_monthly.loc[q1_months, quarter_metrics].mean(),
    'q3_monthly_mean': comparison_monthly.loc[q3_months, quarter_metrics].mean(),
})
quarter_summary['relative_change_pct'] = (
    quarter_summary['q3_monthly_mean'] / quarter_summary['q1_monthly_mean'] - 1
) * 100
display(quarter_summary.round(3))
fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout='constrained')
for ax, metric, title in zip(axes,
                            ['t2_weight_pct', 't9_weight_pct', 'mentions_per_1000'],
                            ['Changing jobs', 'Economy & unemployment', 'Layoff mentions']):
    change = quarter_summary.loc[metric, 'relative_change_pct']
    ax.bar([0], [change], color={'t2_weight_pct': '#087F72', 't9_weight_pct': '#BB6E2B'}.get(metric, '#465568'))
    ax.axhline(0, color='grey', linewidth=0.8)
    ax.text(0, change + (1.5 if change >= 0 else -1.5), f'{change:+.0f}%',
            ha='center', va='bottom' if change >= 0 else 'top', fontsize=18)
    ax.set(title=title, xticks=[], ylim=(-33, 15), ylabel='Relative change, Q1 to Q3 (%)')
fig.savefig(comparison_fig_dir / 'discussion_change_summary.png', dpi=160, bbox_inches='tight')
plt.show()
quarter_summary.to_csv(comparison_output_dir / 'quarterly_layoff_context.csv')

### 7.2 Monthly trajectories and same-month correlations

Plot the monthly layoff-mention rate and disclosed headcount in their original
units. Both y-axes start at zero so that the size of month-to-month changes is not
visually exaggerated. Event count is retained in the comparison table as a
second external measure.

Calculate Spearman correlations between the archive mention rate and each
external measure over the nine complete monthly pairs. Spearman correlation is
used because it compares rank ordering and does not require a linear relationship.
With only nine observations, treat the coefficients and p-values as descriptive;
they do not evaluate forecasting performance or establish a causal relationship.

In [ ]:
same_month_rows = []
for metric in ['disclosed_headcount', 'events']:
    result = spearmanr(comparison_monthly['mentions_per_1000'], comparison_monthly[metric])
    same_month_rows.append({'external_measure': metric, 'n_months': len(comparison_monthly),
                            'spearman_rho': result.statistic, 'p_value': result.pvalue})
same_month_results = pd.DataFrame(same_month_rows).set_index('external_measure')
display(same_month_results.round(4))
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout='constrained')
month_positions = np.arange(1, len(complete_months) + 1)
axes[0].plot(month_positions, comparison_monthly['mentions_per_1000'], '-o', color='#465568')
axes[0].set(ylabel='Mentions per 1,000 comments', ylim=(0, 20), title='r/jobs layoff mentions')
axes[0].yaxis.set_major_locator(MultipleLocator(5))
axes[1].plot(month_positions, comparison_monthly['disclosed_headcount'], '-o', color='#81776D')
axes[1].set(ylabel='Reported employees laid off', ylim=(0, 30000),
             xlabel='2026 month', title='layoffs.fyi disclosed headcount')
axes[1].yaxis.set_major_locator(MultipleLocator(10000))
axes[1].yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
axes[1].set_xticks(month_positions, [month[5:] for month in complete_months])
for ax in axes:
    ax.grid(axis='y', color='#E1E6EB')
    ax.set_axisbelow(True)
fig.savefig(comparison_fig_dir / 'monthly_mentions_and_headcount.png', dpi=160, bbox_inches='tight')
plt.show()
same_month_results.to_csv(comparison_output_dir / 'same_month_correlations.csv')

### 7.3 Recorded event size and missing headcounts

Some layoffs.fyi events do not report the number of employees affected. For each
quarter, sum the disclosed headcount and divide it by the number of events with
known headcount. This gives an event-weighted mean disclosed size rather than an
unweighted average of the three monthly means. Events with missing headcount are
kept missing and are not assigned a value of zero.

Also report the total number of recorded events, the number with known headcount,
the missing-headcount share and the average number of known-headcount events per
month. These diagnostics help assess whether a change in disclosed headcount may
reflect changes in event frequency, recorded event size or data availability.

In [ ]:
event_size_rows = []
for quarter, months in [('Q1', q1_months), ('Q3', q3_months)]:
    quarter_external = external_monthly.loc[months]
    totals = quarter_external[['events', 'known_events', 'missing_events', 'disclosed_headcount']].sum()
    event_size_rows.append({
        'quarter': quarter, 'recorded_events': totals['events'],
        'known_headcount_events': totals['known_events'],
        'disclosed_headcount': totals['disclosed_headcount'],
        'mean_disclosed_size': totals['disclosed_headcount'] / totals['known_events'],
        'missing_headcount_pct': totals['missing_events'] / totals['events'] * 100,
        'known_events_per_month': totals['known_events'] / len(months),
        'all_events_per_month': totals['events'] / len(months),
    })
event_size_summary = pd.DataFrame(event_size_rows).set_index('quarter')
size_change_pct = (event_size_summary.loc['Q3', 'mean_disclosed_size'] /
                   event_size_summary.loc['Q1', 'mean_disclosed_size'] - 1) * 100
display(event_size_summary.round(3))
print(f'Mean disclosed size change: {size_change_pct:.2f}%')
print('Rounded people per event:', event_size_summary['mean_disclosed_size'].round(-1).astype(int).to_dict())

fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout='constrained')
for ax, metric, title, unit in zip(
    axes, ['all_events_per_month', 'mean_disclosed_size', 'missing_headcount_pct'],
    ['Recorded event frequency', 'Disclosed event size', 'Missing headcounts'],
    ['Events per month', 'People per known event', 'Share of recorded events (%)'],
):
    bars = ax.bar(event_size_summary.index, event_size_summary[metric], color=['#AEB7C2', '#465568'])
    ax.bar_label(bars, fmt='%.1f', padding=4)
    ax.set(title=title, ylabel=unit, ylim=(0, event_size_summary[metric].max() * 1.2))
fig.savefig(comparison_fig_dir / 'event_size_and_missingness.png', dpi=160, bbox_inches='tight')
plt.show()
event_size_summary.to_csv(comparison_output_dir / 'quarterly_event_size.csv')

The event-size calculation describes the records available in layoffs.fyi. It
does not estimate the size of events with missing headcount. Comparing the
missing-headcount share and the number of known-headcount events provides a data
quality check, but it cannot rule out selective disclosure of event size.

## 8. Notes
- layoffs.fyi mostly tracks tech companies and dates layoffs by announcement; r/jobs covers every industry and the time people are affected. A weak link between the two is therefore not surprising.
- Only two terms are counted. "severance", "downsizing" and "redundancy" were left out because they are ambiguous (severance also comes up when people resign) or kept timing out in the archive, and "laying off" is not covered.
- The archive is a public copy of Reddit and its totals include deleted and bot comments.
- Nine months of data cannot separate a real relationship from chance; report the curves and correlations as descriptive.